# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/shiva-sn/ML/blob/main/work/notebooks/w03_feature_leakage_check.ipynb?flush_cache=true)

**Lane:** Structured Content Archetype Clustering

This notebook defines the eight-feature vector used by W05 and checks for identifier, future-information, label-derived, query-coverage, and privacy leakage. The model is unsupervised, so there is no prediction target.

## 1. Build the feature vector

W05 uses eight numeric features: search demand, content length, freshness, search visibility, and engagement. IDs are not features. `avg_position_90d = 0` is interpreted as missing, numeric missingness is median-imputed, skewed count/volume fields receive `log1p`, and the final matrix is RobustScaled.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import RobustScaler

DATA_CANDIDATES = [
    Path("../outputs/content_archetypes_clustered.parquet"),
    Path("../outputs/content_level_model_dataset.parquet"),
    Path("../../data/raw/content_refresh_anonymized.csv"),
    Path("../data/raw/content_refresh_anonymized.csv"),
]
DATA_PATH = next((p for p in DATA_CANDIDATES if p.exists()), None)
if DATA_PATH is None:
    raise FileNotFoundError("W05 output or anonymized raw CSV not found.")

model_df = pd.read_parquet(DATA_PATH) if DATA_PATH.suffix == ".parquet" else pd.read_csv(DATA_PATH)
core_features = [
    "search_volume", "word_count", "content_age_days", "days_since_update",
    "impressions_90d", "ctr_90d", "avg_position_90d", "engagement_rate",
]
required = ["content_hash_id", "client_hash_id"] + core_features
missing = [c for c in required if c not in model_df.columns]
if missing:
    raise ValueError(f"Missing required fields: {missing}")

X_raw = model_df[core_features].copy()
zero_position_count = int((X_raw["avg_position_90d"] == 0).sum())
X_raw["avg_position_90d"] = X_raw["avg_position_90d"].replace(0, np.nan)

imputer = SimpleImputer(strategy="median")
X_imputed = pd.DataFrame(imputer.fit_transform(X_raw), columns=core_features, index=X_raw.index)
log_features = ["search_volume", "word_count", "impressions_90d"]
for col in log_features:
    X_imputed[col] = np.log1p(X_imputed[col].clip(lower=0))

scaler = RobustScaler()
X_scaled = scaler.fit_transform(X_imputed)
feature_matrix = pd.DataFrame(X_scaled, columns=core_features, index=model_df.index)

print("Loaded:", DATA_PATH.resolve())
print("Rows:", len(model_df))
print("Feature matrix:", feature_matrix.shape)
print("avg_position_90d zeros converted to missing:", zero_position_count)
print("Missing after imputation:", int(X_imputed.isna().sum().sum()))
display(feature_matrix.head())

## 2. Feature notes

Each feature is observable in the analysis snapshot. This is descriptive clustering, not supervised prediction; `available-when?` therefore means available within the snapshot used to define the archetype.

In [ ]:
feature_notes = pd.DataFrame([
    ("search_volume", "Observed topic search demand", "median", True),
    ("word_count", "Content length", "median + log1p", True),
    ("content_age_days", "Age at snapshot", "median", True),
    ("days_since_update", "Days since recorded update", "median", True),
    ("impressions_90d", "Observed 90-day search impressions", "median + log1p", True),
    ("ctr_90d", "Observed 90-day click-through rate", "median", True),
    ("avg_position_90d", "Observed 90-day average position; zero means no position data", "0→NaN + median", True),
    ("engagement_rate", "Observed engagement rate", "median", True),
], columns=["feature","meaning","missing_handling","snapshot_available"])
display(feature_notes)

## 3. The leakage hunt

The checks attack the vector for identifier leakage, future/trend/label leakage, query-coverage leakage, and privacy exposure. A field may exist in the source table and still be correctly excluded from the model vector.

In [ ]:
id_columns = [c for c in ["client_hash_id","content_hash_id","client_id","content_id"] if c in model_df.columns]
future_keywords = ["future","trend","decline","outcome","label","target"]
future_like_features = [c for c in core_features if any(k in c.lower() for k in future_keywords)]
query_columns = [c for c in model_df.columns if "query" in c.lower()]
query_in_features = sorted(set(query_columns).intersection(core_features))
privacy_columns = [c for c in model_df.columns if any(k in c.lower() for k in ["client_name","company","domain","brand","url"])]
privacy_in_features = sorted(set(privacy_columns).intersection(core_features))

leakage_audit = pd.DataFrame([
    ("Identifier leakage", id_columns, sorted(set(id_columns).intersection(core_features)), "PASS" if not set(id_columns).intersection(core_features) else "FAIL"),
    ("Future/trend/label leakage", future_like_features, future_like_features, "PASS" if not future_like_features else "FAIL"),
    ("Query breadth leakage", query_columns, query_in_features, "PASS" if not query_in_features else "FAIL"),
    ("Privacy-field exposure", privacy_columns, privacy_in_features, "PASS" if not privacy_in_features else "FAIL"),
], columns=["risk","source_columns","in_core_features","status"])
display(leakage_audit)

### Leakage conclusion

The W05 vector is limited to snapshot-level content/search/engagement signals. Identifiers are retained only for joining and interpretation. Query breadth is excluded because missing query coverage can represent data availability rather than a substantive archetype. Future/trend/label-like fields are excluded from the core vector.

In [ ]:
assert "client_hash_id" not in core_features
assert "content_hash_id" not in core_features
assert not future_like_features
assert not query_in_features
assert not privacy_in_features
assert isinstance(imputer, SimpleImputer) and imputer.strategy == "median"
assert isinstance(scaler, RobustScaler)
assert int(X_imputed.isna().sum().sum()) == 0
print("All leakage assertions PASS.")

## 4. What I excluded and why

- `client_hash_id` — identifier only; otherwise clusters could reflect client identity.
- `content_hash_id` — identifier only; unique IDs have no intended behavioral meaning.
- Query breadth / `query_count_90d` — excluded because missing query coverage can create data-availability clusters.
- Future/trend/label-derived fields — excluded to preserve the snapshot-based interpretation.
- Client names, domains, URLs, and similar identifying fields — unnecessary for clustering and inappropriate for paper-facing outputs.
- Provider/model metadata — not part of the content archetype definition.

In [ ]:
exclusions = pd.DataFrame([
    ("client_hash_id", "Identifier only"),
    ("content_hash_id", "Identifier only"),
    ("query breadth / query_count_90d", "Potential data-coverage signal; excluded from W05 core vector"),
    ("future / trend / label fields", "Potential later-information leakage"),
    ("client names / domains / URLs", "Privacy and unnecessary identity information"),
    ("provider / model metadata", "Not part of the archetype definition"),
], columns=["excluded_field_group","reason"])
display(exclusions)

## Self-check

Run top-to-bottom on a fresh Colab/Jupyter runtime. The automated checks cover the mechanical feature-vector and leakage requirements.

In [ ]:
checks = [
    ("Source data loaded", len(model_df) > 0),
    ("Exactly eight core features", len(core_features) == 8),
    ("Identifiers excluded", not set(id_columns).intersection(core_features)),
    ("Future/trend/label fields excluded", not future_like_features),
    ("Query breadth excluded", not query_in_features),
    ("Privacy-like fields excluded", not privacy_in_features),
    ("Median imputation used", isinstance(imputer, SimpleImputer) and imputer.strategy == "median"),
    ("Robust scaling used", isinstance(scaler, RobustScaler)),
    ("No missing values after imputation", int(X_imputed.isna().sum().sum()) == 0),
    ("Feature rows match source rows", len(feature_matrix) == len(model_df)),
]
check_df = pd.DataFrame(checks, columns=["check","passed"])
display(check_df)
if not check_df["passed"].all():
    raise AssertionError("One or more W03 leakage checks failed.")
print("All W03 feature-vector and leakage checks PASS.")